In [3]:
# Install updated LangChain packages
!pip install langchain-core langchain-chroma langchain-huggingface sentence-transformers -q

In [4]:
# Import vector store, embeddings, and document schema from updated paths
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [5]:
# Define sample candidate resumes
resumes = [
    "John Doe: Junior Data Analyst with 2 years experience in SQL, Tableau, and Python data visualization.",
    "Jane Smith: Senior Machine Learning Engineer proficient in PyTorch, LangChain, RAG systems, and MLOps.",
    "Alex Brown: Frontend Developer skilled in React, JavaScript, HTML/CSS, and UI design.",
    "Sarah Connor: NLP Specialist experienced in HuggingFace Transformers, Fine-Tuning LLMs, and SpaCy."
]

# Convert text into LangChain Document format
documents = [Document(page_content=res, metadata={"id": idx}) for idx, res in enumerate(resumes)]

In [6]:
# Load Sentence Transformer embedding model
embedding_function = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Store document embeddings inside Chroma VectorDB
vector_db = Chroma.from_documents(documents, embedding_function)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
# Define recruiter query
query = "Find me a candidate who knows SQL and Tableau"

# Perform top-k similarity search
results = vector_db.similarity_search(query, k=2)

# Display matched candidate resumes
print(f"Recruiter Query: '{query}'\n")
for i, doc in enumerate(results, 1):
    print(f"Match {i}: {doc.page_content}")

Recruiter Query: 'Find me a candidate who knows SQL and Tableau'

Match 1: John Doe: Junior Data Analyst with 2 years experience in SQL, Tableau, and Python data visualization.
Match 2: Jane Smith: Senior Machine Learning Engineer proficient in PyTorch, LangChain, RAG systems, and MLOps.


In [8]:
from transformers import pipeline

# Load a lightweight LLM for text generation
generator = pipeline("text-generation", model="gpt2")

# Create prompt using retrieved context
retrieved_context = results[0].page_content
prompt = f"Context: {retrieved_context}\nQuestion: Summarize why this candidate fits the role.\nAnswer:"

# Generate response using LLM
response = generator(prompt, max_new_tokens=35, num_return_sequences=1)
print(response[0]['generated_text'])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=35) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Context: John Doe: Junior Data Analyst with 2 years experience in SQL, Tableau, and Python data visualization.
Question: Summarize why this candidate fits the role.
Answer: A good candidate for this role will be a programmer with a strong understanding of SQL and the data science area. A good candidate for this role is well versed in the data
